# RideBase Production Data Feasibility — Issue #579

Bu notebook, RideBase production verisinin:

1. sonraki servis zamanı tahmini
2. sonraki bakım task tahmini

için yeterli olup olmadığını değerlendirir. **Model kurmaz.** RideBase Synthetic Dataset v1.2 yalnızca referans taxonomy, beklenen feature contract ve canonical task/category vocabulary için kullanılır; production sonucu olarak yorumlanmaz.

Production export read-only kabul edilir. Notebook hiçbir database üzerinde `UPDATE`, `DELETE`, `INSERT` veya schema migration çalıştırmaz ve PII değerlerini raporlamaz.

## 1. Merkezi config ve veri kaynağı keşfi

Production path hard-code edilmez. Önce opsiyonel `RIDEBASE_PRODUCTION_DATA_ROOT` environment değişkeni, sonra çalışma alanındaki read-only export dosyaları incelenir. Sentetik release klasörleri, notebook çıktıları ve legacy `ridebase-ml/data` kopyaları production adayı değildir.

In [1]:
from pathlib import Path
from difflib import SequenceMatcher
import hashlib
import json
import os
import re
import sqlite3
import unicodedata

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)
sns.set_theme(style="whitegrid", context="notebook")

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "notebooks").is_dir() and (candidate / "reports").is_dir():
            return candidate
    raise FileNotFoundError("ridebase-ml proje kökü bulunamadı")

PROJECT_ROOT = find_project_root()
SYNTHETIC_REFERENCE_ROOT = PROJECT_ROOT.parent / "ridebase_v1_2"
REPORTS_ROOT = PROJECT_ROOT / "reports"
REPORT_TABLES_ROOT = REPORTS_ROOT / "tables"
FIGURES_ROOT = REPORTS_ROOT / "figures" / "production_feasibility"
REPORT_TABLES_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)

REQUIRED_ENTITIES = ["motorcycles", "services", "service_todos", "part_categories"]
OPTIONAL_ENTITIES = ["service_parts", "parts", "workshops", "tenants", "users", "customers"]
SUPPORTED_SUFFIXES = {".csv", ".parquet", ".json", ".sqlite", ".sqlite3", ".db"}

# Açıklanabilir QA/mapping eşikleri; model parametresi değildir.
MAX_ODOMETER_KM = 500_000
MAX_DAILY_KM = 1_000
SUSPICIOUS_KM_JUMP = 100_000
EXTREME_SERVICE_GAP_DAYS = 3_650
FUZZY_AUTO_ACCEPT = 0.90
FUZZY_REVIEW = 0.75
MIN_TITLE_LENGTH = 3

def is_excluded_path(path):
    lowered = {part.lower() for part in path.parts}
    return bool(lowered & {"node_modules", ".git", "reports", "notebooks", "ridebase_v1_1", "ridebase_v1_2", "source_tables", "derived_outputs"})

def discover_production_root():
    explicit = os.getenv("RIDEBASE_PRODUCTION_DATA_ROOT")
    if explicit:
        explicit_path = Path(explicit).expanduser().resolve()
        if explicit_path.exists():
            return explicit_path, "environment:RIDEBASE_PRODUCTION_DATA_ROOT"

    roots = [PROJECT_ROOT, PROJECT_ROOT.parent]
    candidates = {}
    for root in roots:
        for dirpath, dirnames, filenames in os.walk(root):
            current = Path(dirpath)
            depth = len(current.relative_to(root).parts)
            dirnames[:] = [d for d in dirnames if d not in {"node_modules", ".git", "reports", "notebooks", "ridebase_v1_1", "ridebase_v1_2"}]
            if depth > 4 or is_excluded_path(current):
                dirnames[:] = []
                continue
            entity_hits = set()
            db_hits = []
            for name in filenames:
                path = current / name
                if path.suffix.lower() not in SUPPORTED_SUFFIXES:
                    continue
                stem = path.stem.lower()
                for entity in REQUIRED_ENTITIES:
                    if stem == entity or stem.startswith(entity + "_"):
                        entity_hits.add(entity)
                if path.suffix.lower() in {".sqlite", ".sqlite3", ".db"}:
                    db_hits.append(path)
            if {"service_todos", "part_categories"} & entity_hits and len(entity_hits) >= 2:
                candidates[current] = (len(entity_hits), "filesystem entity files")
            for db_path in db_hits:
                try:
                    con = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
                    tables = {row[0].lower() for row in con.execute("SELECT name FROM sqlite_master WHERE type='table'")}
                    con.close()
                    hits = set(REQUIRED_ENTITIES) & tables
                    if {"service_todos", "part_categories"} & hits and len(hits) >= 2:
                        candidates[db_path] = (len(hits), "read-only SQLite tables")
                except sqlite3.Error:
                    pass
    if not candidates:
        return None, "no qualifying local production export"
    best = sorted(candidates.items(), key=lambda item: (-item[1][0], str(item[0])))[0]
    return best[0], best[1][1]

PRODUCTION_DATA_ROOT, PRODUCTION_DISCOVERY_METHOD = discover_production_root()
USE_PRODUCTION_DATA = PRODUCTION_DATA_ROOT is not None
STATUS = "READY" if USE_PRODUCTION_DATA else "BLOCKED"
STATUS_REASON = "Production extract discovered" if USE_PRODUCTION_DATA else "Production extract not available"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SYNTHETIC_REFERENCE_ROOT (taxonomy only):", SYNTHETIC_REFERENCE_ROOT)
print("PRODUCTION_DATA_ROOT:", PRODUCTION_DATA_ROOT)
print("USE_PRODUCTION_DATA:", USE_PRODUCTION_DATA)
print("STATUS:", STATUS)
print("Reason:", STATUS_REASON)
if not USE_PRODUCTION_DATA:
    display(Markdown("## PRODUCTION DATA NOT AVAILABLE\n\n**STATUS = BLOCKED**  \nReason: Production extract not available. Sentetik v1.2 sonuçları production metriği olarak kullanılmayacaktır."))

PROJECT_ROOT: /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase-ml
SYNTHETIC_REFERENCE_ROOT (taxonomy only): /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase_v1_2
PRODUCTION_DATA_ROOT: None
USE_PRODUCTION_DATA: False
STATUS: BLOCKED
Reason: Production extract not available


## PRODUCTION DATA NOT AVAILABLE

**STATUS = BLOCKED**  
Reason: Production extract not available. Sentetik v1.2 sonuçları production metriği olarak kullanılmayacaktır.

## 2. Read-only loader, column discovery ve privacy guard

CSV, Parquet, JSON ve SQLite exportları desteklenir. SQLite yalnızca `mode=ro` ile açılır. Kolonlar semantic alias listesiyle keşfedilir; eşleşmeyen alanlarda analiz `BLOCKED`/`PARTIAL` olur, tahmin edilip crash edilmez.

In [2]:
ENTITY_ALIASES = {
    "motorcycles": ["motorcycles", "motorcycle"],
    "services": ["services", "service"],
    "service_todos": ["service_todos", "service_todo", "servicetodos"],
    "part_categories": ["part_categories", "part_category", "partcategories"],
    "service_parts": ["service_parts", "service_part"],
    "parts": ["parts", "part"],
    "workshops": ["workshops", "workshop", "branches", "branch"],
    "tenants": ["tenants", "tenant", "companies", "company"],
}
SEMANTIC_ALIASES = {
    "motorcycle_id": ["motorcycle_id", "motorcycleid", "bike_id", "vehicle_id", "id"],
    "service_id": ["service_id", "serviceid", "work_order_id", "workorder_id", "id"],
    "todo_id": ["todo_id", "service_todo_id", "servicetodoid", "id"],
    "category_id": ["category_id", "part_category_id", "partcategoryid", "id"],
    "tenant_id": ["tenant_id", "tenantid", "company_id", "companyid", "organization_id", "workshop_id", "branch_id"],
    "service_date": ["service_date", "serviced_at", "service_at", "completed_at", "received_at", "date", "created_at", "createdat"],
    "service_mileage": ["mileage", "odometer", "odometer_km", "kilometer", "service_mileage", "current_mileage"],
    "motorcycle_mileage": ["current_mileage", "mileage", "odometer", "odometer_km", "kilometer"],
    "todo_title": ["title", "task_title", "display_title", "name", "description"],
    "category_name": ["name", "category_name", "title", "display_name", "label"],
    "created_at": ["created_at", "createdat", "inserted_at"],
    "status": ["status", "state"],
    "model": ["model", "model_name", "motorcycle_model"],
}
PII_COLUMN_PATTERNS = re.compile(r"(^|_)(name|email|phone|address|vin|plate|chassis|identity|notes?)(_|$)", re.I)

def canonical_entity_name(name):
    normalized = re.sub(r"[^a-z0-9_]", "", name.lower())
    for entity, aliases in ENTITY_ALIASES.items():
        if normalized in aliases:
            return entity
    return None

def read_tabular_file(path):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".json":
        payload = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(payload, list):
            return pd.json_normalize(payload)
        if isinstance(payload, dict):
            for key in ["data", "rows", "records", "items"]:
                if isinstance(payload.get(key), list):
                    return pd.json_normalize(payload[key])
        raise ValueError(f"Tabular JSON yapısı bulunamadı: {path.name}")
    raise ValueError(f"Desteklenmeyen format: {suffix}")

def load_production_tables(root):
    tables = {}
    sources = {}
    if root is None:
        return tables, sources
    if root.is_file() and root.suffix.lower() in {".sqlite", ".sqlite3", ".db"}:
        con = sqlite3.connect(f"file:{root}?mode=ro", uri=True)
        try:
            available = [row[0] for row in con.execute("SELECT name FROM sqlite_master WHERE type='table'")]
            for raw_name in available:
                entity = canonical_entity_name(raw_name)
                if entity and entity not in tables:
                    safe_table = raw_name.replace('"', '""')
                    tables[entity] = pd.read_sql_query(f'SELECT * FROM "{safe_table}"', con)
                    sources[entity] = f"sqlite:{root.name}:{raw_name}"
        finally:
            con.close()
        return tables, sources

    search_root = root if root.is_dir() else root.parent
    for path in sorted(search_root.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in {".csv", ".parquet", ".json"}:
            continue
        entity = canonical_entity_name(path.stem)
        if entity and entity not in tables:
            try:
                tables[entity] = read_tabular_file(path)
                sources[entity] = str(path)
            except (ValueError, OSError, json.JSONDecodeError) as exc:
                print(f"SKIP {path.name}: {exc}")
    return tables, sources

def discover_column(frame, semantic, exclude=None):
    if frame is None:
        return None
    exclude = set(exclude or [])
    normalized = {re.sub(r"[^a-z0-9]", "", str(col).lower()): col for col in frame.columns if col not in exclude}
    for alias in SEMANTIC_ALIASES[semantic]:
        key = re.sub(r"[^a-z0-9]", "", alias.lower())
        if key in normalized:
            return normalized[key]
    return None

def anonymize_ids(series, prefix="TENANT"):
    values = series.astype("string").fillna("<NULL>")
    unique_values = sorted(values.unique().tolist())
    mapping = {value: f"{prefix}_{index:03d}" for index, value in enumerate(unique_values, 1)}
    return values.map(mapping)

def privacy_safe_columns(frame):
    return [col for col in frame.columns if not PII_COLUMN_PATTERNS.search(str(col))]

production_tables, production_sources = load_production_tables(PRODUCTION_DATA_ROOT)
production_available = all(name in production_tables for name in REQUIRED_ENTITIES)
if USE_PRODUCTION_DATA and not production_available:
    STATUS = "BLOCKED"
    STATUS_REASON = "Production root bulundu ancak minimum dört entity eksik"
print("Loaded production entities:", sorted(production_tables))
print("Minimum production entities available:", production_available)

Loaded production entities: []
Minimum production entities available: False


## 3. Production table inventory ve gerçek kolon keşfi

Her tablo için satır/sütun sayısı, dtype, null, duplicate, primary-key ve foreign-key adayları raporlanır. PII kolonlarının **yalnızca isimleri** schema keşfi için görülebilir; değerleri notebook çıktısına yazdırılmaz.

In [3]:
def candidate_primary_id(frame):
    candidates = []
    for column in frame.columns:
        if str(column).lower() == "id" or str(column).lower().endswith("_id") or str(column).lower().endswith("id"):
            if frame[column].notna().all() and frame[column].is_unique:
                candidates.append(str(column))
    return candidates[0] if candidates else None

def candidate_foreign_ids(frame, primary_id):
    return [str(col) for col in frame.columns if col != primary_id and (str(col).lower().endswith("_id") or str(col).lower().endswith("id"))]

semantic_columns = {}
profile_rows = []
inventory_rows = []
for entity in REQUIRED_ENTITIES + OPTIONAL_ENTITIES:
    frame = production_tables.get(entity)
    if frame is None:
        if entity in REQUIRED_ENTITIES:
            inventory_rows.append({"table_name": entity, "row_count": pd.NA, "column_count": 0, "primary_id": pd.NA, "date_range_start": pd.NaT, "date_range_end": pd.NaT, "tenant_count": pd.NA, "notes": "PRODUCTION DATA NOT AVAILABLE"})
        continue
    primary_id = candidate_primary_id(frame)
    date_semantic = "service_date" if entity == "services" else "created_at"
    date_col = discover_column(frame, date_semantic)
    tenant_col = discover_column(frame, "tenant_id")
    parsed_dates = pd.to_datetime(frame[date_col], errors="coerce") if date_col else pd.Series(dtype="datetime64[ns]")
    inventory_rows.append({
        "table_name": entity, "row_count": len(frame), "column_count": frame.shape[1],
        "primary_id": primary_id, "date_range_start": parsed_dates.min() if len(parsed_dates) else pd.NaT,
        "date_range_end": parsed_dates.max() if len(parsed_dates) else pd.NaT,
        "tenant_count": frame[tenant_col].nunique(dropna=True) if tenant_col else pd.NA,
        "notes": f"source={production_sources.get(entity)}; FK candidates={candidate_foreign_ids(frame, primary_id)}",
    })
    for column in frame.columns:
        profile_rows.append({
            "table_name": entity, "column": str(column), "dtype": str(frame[column].dtype),
            "null_rate": float(frame[column].isna().mean()), "duplicate_values": int(frame[column].duplicated().sum()),
            "is_pii_pattern": bool(PII_COLUMN_PATTERNS.search(str(column))),
        })

production_table_inventory = pd.DataFrame(inventory_rows, columns=["table_name", "row_count", "column_count", "primary_id", "date_range_start", "date_range_end", "tenant_count", "notes"])
production_column_profile = pd.DataFrame(profile_rows)

if production_available:
    semantic_columns = {
        "motorcycle_id@motorcycles": discover_column(production_tables["motorcycles"], "motorcycle_id"),
        "motorcycle_id@services": discover_column(production_tables["services"], "motorcycle_id"),
        "service_id@services": discover_column(production_tables["services"], "service_id"),
        "service_id@service_todos": discover_column(production_tables["service_todos"], "service_id"),
        "tenant_id@motorcycles": discover_column(production_tables["motorcycles"], "tenant_id"),
        "tenant_id@services": discover_column(production_tables["services"], "tenant_id"),
        "service_date": discover_column(production_tables["services"], "service_date"),
        "service_mileage": discover_column(production_tables["services"], "service_mileage"),
        "motorcycle_mileage": discover_column(production_tables["motorcycles"], "motorcycle_mileage"),
        "todo_title": discover_column(production_tables["service_todos"], "todo_title"),
        "part_category_name": discover_column(production_tables["part_categories"], "category_name"),
    }
else:
    semantic_columns = {name: None for name in ["motorcycle_id@motorcycles", "motorcycle_id@services", "service_id@services", "service_id@service_todos", "tenant_id@motorcycles", "tenant_id@services", "service_date", "service_mileage", "motorcycle_mileage", "todo_title", "part_category_name"]}

display(production_table_inventory)
display(pd.DataFrame([{"semantic": key, "discovered_column": value, "status": "FOUND" if value else "BLOCKED"} for key, value in semantic_columns.items()]))
if not production_column_profile.empty:
    display(production_column_profile)

,table_name,row_count,column_count,primary_id,date_range_start,date_range_end,tenant_count,notes
0,motorcycles,<NA>,0,<NA>,NaT,NaT,<NA>,PRODUCTION DATA NOT AVAILABLE
1,services,<NA>,0,<NA>,NaT,NaT,<NA>,PRODUCTION DATA NOT AVAILABLE
2,service_todos,<NA>,0,<NA>,NaT,NaT,<NA>,PRODUCTION DATA NOT AVAILABLE
3,part_categories,<NA>,0,<NA>,NaT,NaT,<NA>,PRODUCTION DATA NOT AVAILABLE


,semantic,discovered_column,status
0,motorcycle_id@motorcycles,None,BLOCKED
1,motorcycle_id@services,None,BLOCKED
2,service_id@services,None,BLOCKED
3,service_id@service_todos,None,BLOCKED
4,tenant_id@motorcycles,None,BLOCKED
5,tenant_id@services,None,BLOCKED
6,service_date,None,BLOCKED
7,service_mileage,None,BLOCKED
8,motorcycle_mileage,None,BLOCKED
9,todo_title,None,BLOCKED


## 4. Motorcycle → service coverage, ardışık servis aralıkları ve cold start

Motorcycle başına 0, 1, 2+, 3+ ve 5+ servis hacmi; gün/km interval QA; cold-start oranları üretilecektir. Production verisi yoksa metrikler hesaplanmaz.

In [4]:
coverage_columns = ["motorcycle_id_anon", "service_count", "coverage_group"]
interval_columns = ["metric", "count", "mean", "median", "p25", "p75", "p90", "p95", "negative_count", "zero_count", "extreme_count"]
motorcycle_service_coverage = pd.DataFrame(columns=coverage_columns)
service_interval_summary = pd.DataFrame(columns=interval_columns)
service_pairs = pd.DataFrame()

if production_available:
    motorcycles = production_tables["motorcycles"].copy()
    services = production_tables["services"].copy()
    mc_id = semantic_columns["motorcycle_id@motorcycles"]
    svc_mc_id = semantic_columns["motorcycle_id@services"]
    svc_id = semantic_columns["service_id@services"]
    svc_date = semantic_columns["service_date"]
    svc_mileage = semantic_columns["service_mileage"]
    required_link_fields = [mc_id, svc_mc_id, svc_id, svc_date]
    if all(required_link_fields):
        counts = services.groupby(svc_mc_id).size()
        coverage = motorcycles[[mc_id]].drop_duplicates().copy()
        coverage["service_count"] = coverage[mc_id].map(counts).fillna(0).astype(int)
        coverage["motorcycle_id_anon"] = anonymize_ids(coverage[mc_id], "MC")
        coverage["coverage_group"] = pd.cut(coverage["service_count"], [-1, 0, 1, 2, 4, np.inf], labels=["0", "1", "2", "3–4", "5+"])
        motorcycle_service_coverage = coverage[["motorcycle_id_anon", "service_count", "coverage_group"]]

        interval_cols = [svc_id, svc_mc_id, svc_date] + ([svc_mileage] if svc_mileage else [])
        service_pairs = services[interval_cols].copy()
        service_pairs[svc_date] = pd.to_datetime(service_pairs[svc_date], errors="coerce")
        service_pairs = service_pairs.dropna(subset=[svc_mc_id, svc_date]).sort_values([svc_mc_id, svc_date, svc_id])
        service_pairs["previous_service_date"] = service_pairs.groupby(svc_mc_id)[svc_date].shift(1)
        service_pairs["days_between_services"] = (service_pairs[svc_date] - service_pairs["previous_service_date"]).dt.total_seconds() / 86400
        if svc_mileage:
            service_pairs[svc_mileage] = pd.to_numeric(service_pairs[svc_mileage], errors="coerce")
            service_pairs["previous_mileage"] = service_pairs.groupby(svc_mc_id)[svc_mileage].shift(1)
            service_pairs["km_between_services"] = service_pairs[svc_mileage] - service_pairs["previous_mileage"]
        else:
            service_pairs["km_between_services"] = np.nan

        rows = []
        for metric, extreme_threshold in [("days_between_services", EXTREME_SERVICE_GAP_DAYS), ("km_between_services", SUSPICIOUS_KM_JUMP)]:
            values = pd.to_numeric(service_pairs[metric], errors="coerce").dropna()
            rows.append({"metric": metric, "count": len(values), "mean": values.mean(), "median": values.median(), "p25": values.quantile(.25), "p75": values.quantile(.75), "p90": values.quantile(.90), "p95": values.quantile(.95), "negative_count": int(values.lt(0).sum()), "zero_count": int(values.eq(0).sum()), "extreme_count": int(values.gt(extreme_threshold).sum())})
        service_interval_summary = pd.DataFrame(rows, columns=interval_columns)

coverage_metrics = {}
if not motorcycle_service_coverage.empty:
    total_mc = len(motorcycle_service_coverage)
    serviced_mc = int(motorcycle_service_coverage["service_count"].ge(1).sum())
    two_plus = int(motorcycle_service_coverage["service_count"].ge(2).sum())
    coverage_metrics = {
        "total_motorcycles": total_mc, "serviced_motorcycles": serviced_mc,
        "motorcycles_2plus": two_plus, "rate_2plus_all": two_plus / total_mc if total_mc else np.nan,
        "rate_2plus_serviced": two_plus / serviced_mc if serviced_mc else np.nan,
        "rate_0": motorcycle_service_coverage["service_count"].eq(0).mean(),
        "rate_1": motorcycle_service_coverage["service_count"].eq(1).mean(),
        "rate_le1": motorcycle_service_coverage["service_count"].le(1).mean(),
    }
display(pd.DataFrame([coverage_metrics]) if coverage_metrics else pd.DataFrame([{"status": "BLOCKED", "reason": STATUS_REASON}]))
display(service_interval_summary)

,status,reason
0,BLOCKED,Production extract not available


,metric,count,mean,median,p25,p75,p90,p95,negative_count,zero_count,extreme_count


## 5. Mileage fizibilitesi ve monotonicity

Servis odometer ve güncel motorcycle mileage alanları ayrı değerlendirilir. Null/fill, zero, negatif, implausible değer, aynı motorcycle-date conflict, düşüş, duplicate mileage ve aşırı günlük kilometre raporlanır. Sentetik %100 doluluk production için varsayım değildir.

In [5]:
mileage_quality_columns = ["scope", "total_rows", "non_null", "null", "fill_rate", "zero_count", "negative_count", "implausible_count", "duplicate_motorcycle_date_conflicts", "monotonic_motorcycles", "non_monotonic_motorcycles", "mileage_decrease_events", "duplicate_mileage_events", "suspicious_jump_events"]
mileage_quality_summary = pd.DataFrame(columns=mileage_quality_columns)
motorcycle_mileage_coverage = pd.DataFrame(columns=["coverage_type", "motorcycle_count", "rate"])

if production_available:
    services = production_tables["services"]
    motorcycles = production_tables["motorcycles"]
    svc_mc = semantic_columns["motorcycle_id@services"]
    svc_date = semantic_columns["service_date"]
    svc_mileage = semantic_columns["service_mileage"]
    mc_mileage = semantic_columns["motorcycle_mileage"]
    rows = []
    for scope, frame, mileage_col in [("services", services, svc_mileage), ("motorcycles", motorcycles, mc_mileage)]:
        if mileage_col:
            values = pd.to_numeric(frame[mileage_col], errors="coerce")
            rows.append({"scope": scope, "total_rows": len(frame), "non_null": int(values.notna().sum()), "null": int(values.isna().sum()), "fill_rate": float(values.notna().mean()), "zero_count": int(values.eq(0).sum()), "negative_count": int(values.lt(0).sum()), "implausible_count": int(values.gt(MAX_ODOMETER_KM).sum()), "duplicate_motorcycle_date_conflicts": pd.NA, "monotonic_motorcycles": pd.NA, "non_monotonic_motorcycles": pd.NA, "mileage_decrease_events": pd.NA, "duplicate_mileage_events": pd.NA, "suspicious_jump_events": pd.NA})
        else:
            rows.append({"scope": scope, "total_rows": len(frame), "non_null": 0, "null": len(frame), "fill_rate": 0.0, "zero_count": pd.NA, "negative_count": pd.NA, "implausible_count": pd.NA, "duplicate_motorcycle_date_conflicts": pd.NA, "monotonic_motorcycles": pd.NA, "non_monotonic_motorcycles": pd.NA, "mileage_decrease_events": pd.NA, "duplicate_mileage_events": pd.NA, "suspicious_jump_events": pd.NA})

    if svc_mc and svc_date and svc_mileage:
        mileage_events = services[[svc_mc, svc_date, svc_mileage]].copy()
        mileage_events[svc_date] = pd.to_datetime(mileage_events[svc_date], errors="coerce")
        mileage_events[svc_mileage] = pd.to_numeric(mileage_events[svc_mileage], errors="coerce")
        mileage_events = mileage_events.sort_values([svc_mc, svc_date])
        mileage_events["delta_km"] = mileage_events.groupby(svc_mc)[svc_mileage].diff()
        mileage_events["delta_days"] = mileage_events.groupby(svc_mc)[svc_date].diff().dt.total_seconds() / 86400
        mileage_events["daily_km"] = mileage_events["delta_km"] / mileage_events["delta_days"].replace(0, np.nan)
        conflict_count = int(mileage_events.groupby([svc_mc, svc_date])[svc_mileage].nunique(dropna=True).gt(1).sum())
        decrease_by_mc = mileage_events.assign(decrease=mileage_events["delta_km"].lt(0)).groupby(svc_mc)["decrease"].any()
        svc_row = rows[0]
        svc_row.update({"duplicate_motorcycle_date_conflicts": conflict_count, "monotonic_motorcycles": int((~decrease_by_mc).sum()), "non_monotonic_motorcycles": int(decrease_by_mc.sum()), "mileage_decrease_events": int(mileage_events["delta_km"].lt(0).sum()), "duplicate_mileage_events": int(mileage_events["delta_km"].eq(0).sum()), "suspicious_jump_events": int((mileage_events["delta_km"].gt(SUSPICIOUS_KM_JUMP) | mileage_events["daily_km"].gt(MAX_DAILY_KM)).sum())})

        per_mc = mileage_events.groupby(svc_mc)[svc_mileage].agg(total="size", non_null="count")
        types = pd.Series(np.select([per_mc["non_null"].eq(0), per_mc["non_null"].eq(per_mc["total"])], ["no_mileage", "all_services_have_mileage"], default="partial_mileage"), index=per_mc.index)
        motorcycle_mileage_coverage = types.value_counts().rename_axis("coverage_type").reset_index(name="motorcycle_count")
        motorcycle_mileage_coverage["rate"] = motorcycle_mileage_coverage["motorcycle_count"] / motorcycle_mileage_coverage["motorcycle_count"].sum()
    mileage_quality_summary = pd.DataFrame(rows, columns=mileage_quality_columns)

display(mileage_quality_summary)
display(motorcycle_mileage_coverage)

,scope,total_rows,non_null,null,fill_rate,zero_count,negative_count,implausible_count,duplicate_motorcycle_date_conflicts,monotonic_motorcycles,non_monotonic_motorcycles,mileage_decrease_events,duplicate_mileage_events,suspicious_jump_events


,coverage_type,motorcycle_count,rate


## 6. Censoring tanımı

**A — Snapshot-level censoring:** Bir service snapshot'ın aynı motorcycle için sonraki servisi yoksa censored'dır. Oran, servis snapshotları arasındaki label gözlenebilirliğini gösterir.

**B — Motorcycle-level censoring:** Observation window sonunda servis görmüş her motorcycle'ın en son servis event'i doğal olarak sağdan censored'dır. Bu, snapshot censoring oranıyla aynı denominator veya anlam değildir; son event'in takip penceresi içinde geleceğinin henüz gözlenmediğini anlatır.

In [6]:
censoring_columns = ["unit", "total", "observed", "censored", "censoring_rate", "definition"]
censoring_summary = pd.DataFrame(columns=censoring_columns)
if production_available and not service_pairs.empty:
    svc_mc = semantic_columns["motorcycle_id@services"]
    total_snapshots = len(service_pairs)
    observed = int(service_pairs.groupby(svc_mc).cumcount(ascending=False).gt(0).sum())
    censored = total_snapshots - observed
    serviced_motorcycles = int(service_pairs[svc_mc].nunique())
    censoring_summary = pd.DataFrame([
        {"unit": "service_snapshot", "total": total_snapshots, "observed": observed, "censored": censored, "censoring_rate": censored / total_snapshots if total_snapshots else np.nan, "definition": "snapshot has no later service in extract"},
        {"unit": "serviced_motorcycle_latest_event", "total": serviced_motorcycles, "observed": 0, "censored": serviced_motorcycles, "censoring_rate": 1.0 if serviced_motorcycles else np.nan, "definition": "latest service per serviced motorcycle is naturally right-censored at observation end"},
    ])
display(censoring_summary if not censoring_summary.empty else pd.DataFrame([{"status": "BLOCKED", "reason": STATUS_REASON}]))

,status,reason
0,BLOCKED,Production extract not available


## 7. Tenant/workshop feasibility

Tenant kimlikleri deterministic olarak `TENANT_001` biçiminde anonimleştirilir. Gerçek şirket/workshop adları, müşteri verisi veya diğer PII rapora çıkarılmaz.

In [7]:
tenant_columns = ["tenant_id", "motorcycle_count", "service_count", "service_todo_count", "part_category_count", "rate_2plus_motorcycles", "service_mileage_fill_rate", "todo_title_null_rate", "unique_todo_title_count"]
tenant_feasibility_summary = pd.DataFrame(columns=tenant_columns)
if production_available:
    motorcycles = production_tables["motorcycles"].copy()
    services = production_tables["services"].copy()
    todos = production_tables["service_todos"].copy()
    categories = production_tables["part_categories"].copy()
    mc_tenant = semantic_columns["tenant_id@motorcycles"]
    svc_tenant = semantic_columns["tenant_id@services"]
    todo_tenant = discover_column(todos, "tenant_id")
    cat_tenant = discover_column(categories, "tenant_id")
    if svc_tenant:
        tenant_values = services[svc_tenant]
        anon_map_source = pd.Series(sorted(tenant_values.astype("string").fillna("<NULL>").unique()))
        anon_labels = anonymize_ids(anon_map_source, "TENANT")
        tenant_map = dict(zip(anon_map_source, anon_labels))
        def tenant_anon(frame, col):
            return frame[col].astype("string").fillna("<NULL>").map(tenant_map).fillna("TENANT_UNKNOWN") if col else pd.Series("TENANT_UNKNOWN", index=frame.index)
        motorcycles["tenant_id"] = tenant_anon(motorcycles, mc_tenant)
        services["tenant_id"] = tenant_anon(services, svc_tenant)
        todos["tenant_id"] = tenant_anon(todos, todo_tenant)
        categories["tenant_id"] = tenant_anon(categories, cat_tenant)
        svc_mc = semantic_columns["motorcycle_id@services"]
        mc_id = semantic_columns["motorcycle_id@motorcycles"]
        svc_mileage = semantic_columns["service_mileage"]
        todo_title = semantic_columns["todo_title"]
        service_count_by_mc = services.groupby(svc_mc).size() if svc_mc else pd.Series(dtype=int)
        motorcycles["service_count"] = motorcycles[mc_id].map(service_count_by_mc).fillna(0) if mc_id else 0
        rows = []
        for tenant_id in sorted(set(services["tenant_id"]) | set(motorcycles["tenant_id"])):
            m = motorcycles[motorcycles["tenant_id"].eq(tenant_id)]
            s = services[services["tenant_id"].eq(tenant_id)]
            t = todos[todos["tenant_id"].eq(tenant_id)]
            c = categories[categories["tenant_id"].eq(tenant_id)]
            rows.append({"tenant_id": tenant_id, "motorcycle_count": len(m), "service_count": len(s), "service_todo_count": len(t), "part_category_count": len(c), "rate_2plus_motorcycles": float(m["service_count"].ge(2).mean()) if len(m) else np.nan, "service_mileage_fill_rate": float(pd.to_numeric(s[svc_mileage], errors="coerce").notna().mean()) if svc_mileage and len(s) else np.nan, "todo_title_null_rate": float(t[todo_title].isna().mean()) if todo_title and len(t) else np.nan, "unique_todo_title_count": int(t[todo_title].nunique(dropna=True)) if todo_title else pd.NA})
        tenant_feasibility_summary = pd.DataFrame(rows, columns=tenant_columns)
display(tenant_feasibility_summary)

,tenant_id,motorcycle_count,service_count,service_todo_count,part_category_count,rate_2plus_motorcycles,service_mileage_fill_rate,todo_title_null_rate,unique_todo_title_count


## 8. Part category inventory ve candidate mapping

Production kategorileri lowercase/trim/Türkçe→ASCII/punctuation/whitespace normalizasyonundan geçer. Sentetik v1.2 `part_category_hint` ve `service_parts.part_category` yalnızca canonical vocabulary referansıdır; production category zorla map edilmez.

In [8]:
TURKISH_ASCII = str.maketrans({"ı": "i", "ğ": "g", "ü": "u", "ş": "s", "ö": "o", "ç": "c", "İ": "i", "Ğ": "g", "Ü": "u", "Ş": "s", "Ö": "o", "Ç": "c"})
def normalize_text(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).lower().strip().translate(TURKISH_ASCII)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

canonical_tasks = pd.read_csv(SYNTHETIC_REFERENCE_ROOT / "source_tables" / "maintenance_tasks.csv", encoding="utf-8-sig")
synthetic_parts = pd.read_csv(SYNTHETIC_REFERENCE_ROOT / "source_tables" / "service_parts.csv", usecols=["part_category"], encoding="utf-8-sig")
canonical_part_categories = sorted(set(canonical_tasks["part_category_hint"].dropna().astype(str)) | set(synthetic_parts["part_category"].dropna().astype(str)))
canonical_part_lookup = {normalize_text(value): value for value in canonical_part_categories}

part_inventory_columns = ["tenant_id", "raw_category", "normalized_category", "count"]
part_mapping_columns = ["tenant_id", "raw_category", "normalized_category", "count", "candidate_canonical_category", "match_method", "confidence", "manual_review_required", "mapping_status"]
production_part_category_inventory = pd.DataFrame(columns=part_inventory_columns)
part_category_mapping_candidates = pd.DataFrame(columns=part_mapping_columns)

if production_available:
    categories = production_tables["part_categories"].copy()
    cat_name = semantic_columns["part_category_name"]
    cat_tenant = discover_column(categories, "tenant_id")
    categories["tenant_id"] = anonymize_ids(categories[cat_tenant], "TENANT") if cat_tenant else "TENANT_ALL"
    if cat_name:
        categories["raw_category"] = categories[cat_name].astype("string")
        categories["normalized_category"] = categories["raw_category"].map(normalize_text)
        production_part_category_inventory = categories.dropna(subset=["raw_category"]).groupby(["tenant_id", "raw_category", "normalized_category"], dropna=False).size().rename("count").reset_index()
        mapping_rows = []
        for row in production_part_category_inventory.itertuples(index=False):
            exact = canonical_part_lookup.get(row.normalized_category)
            mapping_rows.append({"tenant_id": row.tenant_id, "raw_category": row.raw_category, "normalized_category": row.normalized_category, "count": row.count, "candidate_canonical_category": exact, "match_method": "NORMALIZED_EXACT" if exact else "UNMAPPED", "confidence": 1.0 if exact else np.nan, "manual_review_required": not bool(exact), "mapping_status": "AUTO_ACCEPT" if exact else "UNMAPPED"})
        part_category_mapping_candidates = pd.DataFrame(mapping_rows, columns=part_mapping_columns)
display(production_part_category_inventory)
display(part_category_mapping_candidates)

,tenant_id,raw_category,normalized_category,count


,tenant_id,raw_category,normalized_category,count,candidate_canonical_category,match_method,confidence,manual_review_required,mapping_status


## 9. Service todos title inventory, label quality ve canonical task mapping

Normalization aggressive stemming yapmaz. Pipeline: normalized exact → sınırlı/açıklanabilir rule → fuzzy similarity → low-confidence manual review. Embedding/LLM bu notebookta label truth üretmez; yalnız ileride review assist olabilir.

In [9]:
todo_inventory_columns = ["tenant_id", "raw_title", "normalized_title", "count", "first_seen", "last_seen", "linked_service_count", "linked_motorcycle_count"]
todo_mapping_columns = ["tenant_id", "raw_title", "normalized_title", "count", "candidate_task_code", "candidate_task_name", "component_group", "match_method", "confidence", "mapping_status", "manual_review_required"]
production_service_todo_title_inventory = pd.DataFrame(columns=todo_inventory_columns)
production_service_todo_mapping_candidates = pd.DataFrame(columns=todo_mapping_columns)
label_quality_summary = pd.DataFrame(columns=["issue", "row_count", "row_rate"])

canonical_task_reference = canonical_tasks[["task_code", "canonical_name_tr", "component_group", "is_fault_based", "is_periodic"]].copy()
canonical_task_reference["normalized_title"] = canonical_task_reference["canonical_name_tr"].map(normalize_text)
canonical_by_title = canonical_task_reference.drop_duplicates("normalized_title").set_index("normalized_title").to_dict("index")
RULES = [
    (lambda title: "motor yagi" in title and "degis" in title, "ENGINE_OIL_CHANGE"),
    (lambda title: "yag filtre" in title and "degis" in title, "OIL_FILTER_CHANGE"),
    (lambda title: "hava filtre" in title and "degis" in title, "AIR_FILTER_CHANGE"),
    (lambda title: "aku" in title and ("test" in title or "kontrol" in title), "BATTERY_TEST"),
    (lambda title: "fren hidro" in title and "degis" in title, "BRAKE_FLUID_CHANGE"),
]

def map_title(normalized_title):
    if not normalized_title:
        return {"candidate_task_code": pd.NA, "candidate_task_name": pd.NA, "component_group": pd.NA, "match_method": "UNMAPPED", "confidence": np.nan, "mapping_status": "UNMAPPED", "manual_review_required": True}
    if normalized_title in canonical_by_title:
        match = canonical_by_title[normalized_title]
        return {"candidate_task_code": match["task_code"], "candidate_task_name": match["canonical_name_tr"], "component_group": match["component_group"], "match_method": "NORMALIZED_EXACT", "confidence": 1.0, "mapping_status": "AUTO_ACCEPT", "manual_review_required": False}
    for predicate, task_code in RULES:
        if predicate(normalized_title):
            match = canonical_task_reference.loc[canonical_task_reference["task_code"].eq(task_code)].iloc[0]
            return {"candidate_task_code": task_code, "candidate_task_name": match["canonical_name_tr"], "component_group": match["component_group"], "match_method": "RULE", "confidence": .92, "mapping_status": "AUTO_ACCEPT", "manual_review_required": False}
    scores = canonical_task_reference["normalized_title"].map(lambda candidate: SequenceMatcher(None, normalized_title, candidate).ratio())
    best_index = scores.idxmax()
    best_score = float(scores.loc[best_index])
    best = canonical_task_reference.loc[best_index]
    if best_score >= FUZZY_AUTO_ACCEPT:
        status, review = "AUTO_ACCEPT", False
    elif best_score >= FUZZY_REVIEW:
        status, review = "REVIEW", True
    else:
        status, review = "UNMAPPED", True
    include_candidate = status != "UNMAPPED"
    return {"candidate_task_code": best["task_code"] if include_candidate else pd.NA, "candidate_task_name": best["canonical_name_tr"] if include_candidate else pd.NA, "component_group": best["component_group"] if include_candidate else pd.NA, "match_method": "FUZZY" if include_candidate else "UNMAPPED", "confidence": best_score, "mapping_status": status, "manual_review_required": review}

todo_metrics = {}
if production_available:
    todos = production_tables["service_todos"].copy()
    services = production_tables["services"].copy()
    todo_title = semantic_columns["todo_title"]
    todo_service = semantic_columns["service_id@service_todos"]
    service_id = semantic_columns["service_id@services"]
    service_mc = semantic_columns["motorcycle_id@services"]
    todo_tenant = discover_column(todos, "tenant_id")
    todo_created = discover_column(todos, "created_at")
    todos["tenant_id"] = anonymize_ids(todos[todo_tenant], "TENANT") if todo_tenant else "TENANT_ALL"
    if todo_title:
        todos["raw_title"] = todos[todo_title].astype("string")
        todos["normalized_title"] = todos["raw_title"].map(normalize_text)
        todos["event_at"] = pd.to_datetime(todos[todo_created], errors="coerce") if todo_created else pd.NaT
        if todo_service and service_id:
            link = services[[service_id] + ([service_mc] if service_mc else [])].drop_duplicates(service_id)
            todos = todos.merge(link, left_on=todo_service, right_on=service_id, how="left", suffixes=("", "_service"))
        group_cols = ["tenant_id", "raw_title", "normalized_title"]
        agg = {"count": ("raw_title", "size"), "first_seen": ("event_at", "min"), "last_seen": ("event_at", "max")}
        if todo_service:
            agg["linked_service_count"] = (todo_service, "nunique")
        if service_mc:
            agg["linked_motorcycle_count"] = (service_mc, "nunique")
        production_service_todo_title_inventory = todos.dropna(subset=["raw_title"]).groupby(group_cols, dropna=False).agg(**agg).reset_index()
        for missing_col in todo_inventory_columns:
            if missing_col not in production_service_todo_title_inventory:
                production_service_todo_title_inventory[missing_col] = pd.NA
        production_service_todo_title_inventory = production_service_todo_title_inventory[todo_inventory_columns]

        mapping_rows = []
        for row in production_service_todo_title_inventory.itertuples(index=False):
            mapped = map_title(row.normalized_title)
            mapping_rows.append({"tenant_id": row.tenant_id, "raw_title": row.raw_title, "normalized_title": row.normalized_title, "count": row.count, **mapped})
        production_service_todo_mapping_candidates = pd.DataFrame(mapping_rows, columns=todo_mapping_columns)

        normalized = todos["normalized_title"].fillna("")
        multi_action = normalized.str.contains(r"\+|,|/|ve|ile", regex=True)
        generic = normalized.isin({"bakim", "kontrol", "islem", "degisim", "onarim"})
        numeric_only = normalized.str.fullmatch(r"\d+").fillna(False)
        too_short = normalized.str.len().lt(MIN_TITLE_LENGTH)
        blank = normalized.eq("")
        issue_masks = {"blank_or_null": blank, "too_short": too_short, "numeric_only": numeric_only, "generic_title": generic, "possible_multi_action": multi_action}
        label_quality_summary = pd.DataFrame([{"issue": issue, "row_count": int(mask.sum()), "row_rate": float(mask.mean())} for issue, mask in issue_masks.items()])
        title_counts = todos["raw_title"].value_counts(dropna=True)
        todo_metrics = {"total_rows": len(todos), "null_title": int(todos[todo_title].isna().sum()), "blank_title": int(blank.sum()), "raw_unique": int(todos["raw_title"].nunique(dropna=True)), "normalized_unique": int(todos["normalized_title"].nunique(dropna=True)), "unique_total_ratio": float(todos["raw_title"].nunique(dropna=True) / len(todos)) if len(todos) else np.nan, "singletons": int(title_counts.eq(1).sum()), "singleton_rate": float(title_counts.eq(1).mean()) if len(title_counts) else np.nan, "titles_ge5": int(title_counts.ge(5).sum()), "titles_ge10": int(title_counts.ge(10).sum()), "titles_ge50": int(title_counts.ge(50).sum()), "raw_variants_merged": int(todos["raw_title"].nunique(dropna=True) - todos["normalized_title"].nunique(dropna=True))}
display(pd.DataFrame([todo_metrics]) if todo_metrics else pd.DataFrame([{"status": "BLOCKED", "reason": STATUS_REASON}]))
display(label_quality_summary)
display(production_service_todo_mapping_candidates.sort_values("count", ascending=False).head(100) if not production_service_todo_mapping_candidates.empty else production_service_todo_mapping_candidates)

,status,reason
0,BLOCKED,Production extract not available


,issue,row_count,row_rate


,tenant_id,raw_title,normalized_title,count,candidate_task_code,candidate_task_name,component_group,match_method,confidence,mapping_status,manual_review_required


## 10. Mapping coverage — row-weighted, unique-title ve tenant bazlı

Sık başlıkların satır kapsamı ile taxonomy'nin farklı unique title'ları kapsama yeteneği ayrı ölçülür. `AUTO_ACCEPT`, `REVIEW` ve `UNMAPPED` oranları ayrı tutulur.

In [10]:
mapping_coverage_columns = ["scope", "tenant_id", "mapping_status", "row_count", "row_rate", "unique_title_count", "unique_title_rate"]
production_mapping_coverage = pd.DataFrame(columns=mapping_coverage_columns)
if not production_service_todo_mapping_candidates.empty:
    rows = []
    def append_coverage(scope, tenant_id, frame):
        total_rows = frame["count"].sum()
        total_unique = len(frame)
        for status in ["AUTO_ACCEPT", "REVIEW", "UNMAPPED"]:
            subset = frame[frame["mapping_status"].eq(status)]
            rows.append({"scope": scope, "tenant_id": tenant_id, "mapping_status": status, "row_count": int(subset["count"].sum()), "row_rate": float(subset["count"].sum() / total_rows) if total_rows else np.nan, "unique_title_count": len(subset), "unique_title_rate": len(subset) / total_unique if total_unique else np.nan})
    append_coverage("GLOBAL", "ALL", production_service_todo_mapping_candidates)
    for tenant_id, frame in production_service_todo_mapping_candidates.groupby("tenant_id"):
        append_coverage("TENANT", tenant_id, frame)
    production_mapping_coverage = pd.DataFrame(rows, columns=mapping_coverage_columns)
display(production_mapping_coverage)

,scope,tenant_id,mapping_status,row_count,row_rate,unique_title_count,unique_title_rate


## 11. Feasibility verdicts, rule fallback ve decision matrix

Verdictler yalnızca production metriklerinden türetilir. Production extract yokken hem next-service hem next-task `NOT_FEASIBLE`, Stage 1 `BLOCKED` kalır. LLM hiçbir zaman label truth üreticisi değildir.

In [11]:
def metric_value(frame, scope, column, default=np.nan):
    if frame.empty or scope not in frame["scope"].values:
        return default
    return frame.loc[frame["scope"].eq(scope), column].iloc[0]

if production_available:
    service_date_fill = pd.to_datetime(production_tables["services"][semantic_columns["service_date"]], errors="coerce").notna().mean() if semantic_columns["service_date"] else 0.0
    service_mileage_fill = float(mileage_quality_summary.loc[mileage_quality_summary["scope"].eq("services"), "fill_rate"].iloc[0]) if "services" in mileage_quality_summary["scope"].values else 0.0
    motorcycle_mileage_fill = float(mileage_quality_summary.loc[mileage_quality_summary["scope"].eq("motorcycles"), "fill_rate"].iloc[0]) if "motorcycles" in mileage_quality_summary["scope"].values else 0.0
    two_plus_rate = coverage_metrics.get("rate_2plus_all", 0.0)
    high_row = production_mapping_coverage.query("scope == 'GLOBAL' and mapping_status == 'AUTO_ACCEPT'")
    mapped_row_rate = float(high_row["row_rate"].iloc[0]) if len(high_row) else 0.0
    mapped_unique_rate = float(high_row["unique_title_rate"].iloc[0]) if len(high_row) else 0.0
    title_null_rate = (todo_metrics.get("null_title", 0) + todo_metrics.get("blank_title", 0)) / todo_metrics.get("total_rows", 1)
    next_service_verdict = "FEASIBLE" if two_plus_rate >= .50 and service_date_fill >= .95 and service_mileage_fill >= .80 else ("PARTIALLY_FEASIBLE" if two_plus_rate >= .20 and service_date_fill >= .80 else "NOT_FEASIBLE")
    next_task_verdict = "FEASIBLE" if mapped_row_rate >= .80 and mapped_unique_rate >= .60 and title_null_rate <= .05 else ("PARTIALLY_FEASIBLE" if mapped_row_rate >= .50 and title_null_rate <= .20 else "NOT_FEASIBLE")
    stage1_status = "COMPLETE" if next_service_verdict != "NOT_FEASIBLE" and next_task_verdict != "NOT_FEASIBLE" else "PARTIAL"
else:
    service_date_fill = service_mileage_fill = motorcycle_mileage_fill = two_plus_rate = mapped_row_rate = mapped_unique_rate = title_null_rate = np.nan
    next_service_verdict = "NOT_FEASIBLE"
    next_task_verdict = "NOT_FEASIBLE"
    stage1_status = "BLOCKED"

rule_fields = {
    "model": semantic_columns.get("model") or (discover_column(production_tables.get("motorcycles"), "model") if production_available else None),
    "last_service_date": semantic_columns.get("service_date"),
    "last_service_mileage": semantic_columns.get("service_mileage"),
    "current_motorcycle_mileage": semantic_columns.get("motorcycle_mileage"),
    "estimated_annual_km": None,
    "maintenance_policy": None,
}
rule_fallback_readiness = "PARTIAL" if production_available and sum(value is not None for value in rule_fields.values()) >= 3 else "BLOCKED"

decision_matrix = pd.DataFrame([
    {"problem": "Next service timing", "data_quality": "BLOCKED" if not production_available else next_service_verdict, "recommended_approach": "Survival/time-to-event ML after leakage-safe snapshot contract; rule fallback for sparse history"},
    {"problem": "Cold start", "data_quality": "BLOCKED" if not production_available else ("WEAK" if coverage_metrics.get("rate_le1", 1) > .30 else "MEDIUM"), "recommended_approach": "Model maintenance policy + last date/mileage + annual-km rule fallback"},
    {"problem": "Todo normalization", "data_quality": "BLOCKED" if not production_available else next_task_verdict, "recommended_approach": "Deterministic normalization + exact/rule + fuzzy/embedding review"},
    {"problem": "Next tasks", "data_quality": "BLOCKED" if not production_available else next_task_verdict, "recommended_approach": "Multi-label ML only after canonical mapping quality is accepted"},
    {"problem": "Low-confidence text", "data_quality": "WEAK", "recommended_approach": "Manual review; optional LLM assist, never label truth"},
])
display(decision_matrix)

,problem,data_quality,recommended_approach
0,Next service timing,BLOCKED,Survival/time-to-event ML after leakage-safe snapshot contract; rule fallback for sparse history
1,Cold start,BLOCKED,Model maintenance policy + last date/mileage + annual-km rule fallback
2,Todo normalization,BLOCKED,Deterministic normalization + exact/rule + fuzzy/embedding review
3,Next tasks,BLOCKED,Multi-label ML only after canonical mapping quality is accepted
4,Low-confidence text,WEAK,"Manual review; optional LLM assist, never label truth"


## 12. Production feasibility scorecard

Her alan `PASS`, `WARN` veya `FAIL` ve kısa kanıtla raporlanır. Veri yokluğu `FAIL`/`BLOCKED` olarak korunur; sentetik referans bunu yükseltmez.

In [12]:
scorecard_columns = ["dimension", "status", "reason"]
if production_available:
    production_feasibility_scorecard = pd.DataFrame([
        {"dimension": "service_history", "status": "PASS" if two_plus_rate >= .50 else ("WARN" if two_plus_rate >= .20 else "FAIL"), "reason": f">=2 service motorcycle rate={two_plus_rate:.2%}"},
        {"dimension": "mileage_quality", "status": "PASS" if service_mileage_fill >= .80 else ("WARN" if service_mileage_fill >= .50 else "FAIL"), "reason": f"service mileage fill={service_mileage_fill:.2%}"},
        {"dimension": "censoring_readiness", "status": "PASS" if not censoring_summary.empty else "FAIL", "reason": "snapshot and motorcycle definitions computed" if not censoring_summary.empty else "cannot compute"},
        {"dimension": "todo_label_quality", "status": "PASS" if title_null_rate <= .05 else ("WARN" if title_null_rate <= .20 else "FAIL"), "reason": f"null+blank rate={title_null_rate:.2%}"},
        {"dimension": "category_mapping_readiness", "status": "PASS" if not part_category_mapping_candidates.empty and part_category_mapping_candidates["mapping_status"].eq("AUTO_ACCEPT").mean() >= .70 else "WARN", "reason": "normalized exact reference coverage evaluated"},
        {"dimension": "tenant_balance", "status": "WARN", "reason": "tenant volume dispersion requires review"},
        {"dimension": "cold_start", "status": "PASS" if coverage_metrics.get("rate_le1", 1) <= .30 else "WARN", "reason": f"<=1 service rate={coverage_metrics.get('rate_le1', np.nan):.2%}"},
        {"dimension": "next_service_target_readiness", "status": "PASS" if next_service_verdict == "FEASIBLE" else ("WARN" if next_service_verdict == "PARTIALLY_FEASIBLE" else "FAIL"), "reason": next_service_verdict},
        {"dimension": "next_task_target_readiness", "status": "PASS" if next_task_verdict == "FEASIBLE" else ("WARN" if next_task_verdict == "PARTIALLY_FEASIBLE" else "FAIL"), "reason": next_task_verdict},
    ], columns=scorecard_columns)
else:
    production_feasibility_scorecard = pd.DataFrame([{"dimension": dimension, "status": "FAIL", "reason": "BLOCKED — Production extract not available"} for dimension in ["service_history", "mileage_quality", "censoring_readiness", "todo_label_quality", "category_mapping_readiness", "tenant_balance", "cold_start", "next_service_target_readiness", "next_task_target_readiness"]], columns=scorecard_columns)
display(production_feasibility_scorecard)

,dimension,status,reason
0,service_history,FAIL,BLOCKED — Production extract not available
1,mileage_quality,FAIL,BLOCKED — Production extract not available
2,censoring_readiness,FAIL,BLOCKED — Production extract not available
3,todo_label_quality,FAIL,BLOCKED — Production extract not available
4,category_mapping_readiness,FAIL,BLOCKED — Production extract not available
5,tenant_balance,FAIL,BLOCKED — Production extract not available
6,cold_start,FAIL,BLOCKED — Production extract not available
7,next_service_target_readiness,FAIL,BLOCKED — Production extract not available
8,next_task_target_readiness,FAIL,BLOCKED — Production extract not available


## 13. Grafikler

Production verisi mevcutsa gerçek dağılımlar çizilir. Veri yoksa istenen dosya adlarında yalnızca açık availability-status görselleri üretilir; bunlar metrik veya fake analiz içermez.

In [13]:
figure_names = [
    "01_services_per_motorcycle.png", "02_days_between_services.png", "03_km_between_services.png",
    "04_service_mileage_missingness.png", "05_todo_title_frequency.png", "06_mapping_coverage.png",
    "07_tenant_service_volume.png", "08_tenant_mapping_coverage.png",
]
def save_status_figure(file_name, subtitle):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.axis("off")
    ax.text(.5, .62, "PRODUCTION DATA NOT AVAILABLE", ha="center", va="center", fontsize=22, weight="bold", color="#B42318")
    ax.text(.5, .40, subtitle, ha="center", va="center", fontsize=12, color="#475467")
    ax.text(.5, .22, "STATUS = BLOCKED — no synthetic metrics substituted", ha="center", va="center", fontsize=11, color="#667085")
    fig.tight_layout()
    fig.savefig(FIGURES_ROOT / file_name, dpi=160, bbox_inches="tight", facecolor="white")
    plt.close(fig)

if not production_available:
    for file_name in figure_names:
        save_status_figure(file_name, file_name.replace("_", " ").replace(".png", ""))
else:
    # 01 services per motorcycle
    fig, ax = plt.subplots(figsize=(12, 6)); sns.histplot(motorcycle_service_coverage["service_count"], discrete=True, ax=ax, color="#175CD3"); ax.set(title="Services per motorcycle", xlabel="Service count", ylabel="Motorcycles"); fig.tight_layout(); fig.savefig(FIGURES_ROOT / figure_names[0], dpi=160); plt.close(fig)
    for column, file_name, title in [("days_between_services", figure_names[1], "Days between services"), ("km_between_services", figure_names[2], "Km between services")]:
        fig, ax = plt.subplots(figsize=(12, 6)); values = pd.to_numeric(service_pairs[column], errors="coerce").dropna(); sns.histplot(values.clip(upper=values.quantile(.99)) if len(values) else values, bins=40, ax=ax, color="#175CD3"); ax.set(title=title, xlabel=column, ylabel="Pairs"); fig.tight_layout(); fig.savefig(FIGURES_ROOT / file_name, dpi=160); plt.close(fig)
    fig, ax = plt.subplots(figsize=(10, 5)); sns.barplot(data=mileage_quality_summary, x="scope", y="fill_rate", ax=ax, color="#12B76A"); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set(title="Mileage fill rate", xlabel="Scope", ylabel="Fill rate"); fig.tight_layout(); fig.savefig(FIGURES_ROOT / figure_names[3], dpi=160); plt.close(fig)
    top_titles = production_service_todo_title_inventory.sort_values("count", ascending=False).head(30); fig, ax = plt.subplots(figsize=(12, 9)); sns.barplot(data=top_titles, x="count", y="raw_title", ax=ax, color="#175CD3"); ax.set(title="Top production todo titles", xlabel="Rows", ylabel="Title"); fig.tight_layout(); fig.savefig(FIGURES_ROOT / figure_names[4], dpi=160); plt.close(fig)
    global_cov = production_mapping_coverage.query("scope == 'GLOBAL'"); fig, ax = plt.subplots(figsize=(10, 5)); sns.barplot(data=global_cov, x="mapping_status", y="row_rate", ax=ax, color="#7F56D9"); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set(title="Row-weighted mapping coverage", xlabel="Status", ylabel="Row rate"); fig.tight_layout(); fig.savefig(FIGURES_ROOT / figure_names[5], dpi=160); plt.close(fig)
    fig, ax = plt.subplots(figsize=(12, 6)); sns.barplot(data=tenant_feasibility_summary.sort_values("service_count", ascending=False), x="service_count", y="tenant_id", ax=ax, color="#175CD3"); ax.set(title="Tenant service volume", xlabel="Services", ylabel="Anonymized tenant"); fig.tight_layout(); fig.savefig(FIGURES_ROOT / figure_names[6], dpi=160); plt.close(fig)
    tenant_cov = production_mapping_coverage.query("scope == 'TENANT' and mapping_status == 'AUTO_ACCEPT'"); fig, ax = plt.subplots(figsize=(12, 6)); sns.barplot(data=tenant_cov, x="row_rate", y="tenant_id", ax=ax, color="#12B76A"); ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set(title="Tenant high-confidence mapping coverage", xlabel="Mapped row rate", ylabel="Anonymized tenant"); fig.tight_layout(); fig.savefig(FIGURES_ROOT / figure_names[7], dpi=160); plt.close(fig)
print("Figure files:", len(list(FIGURES_ROOT.glob("*.png"))))

Figure files: 8


## 14. CSV outputs ve production feasibility report

CSV'ler production verisi yokken de açık şemalarıyla yazılır; boş değerler veri yokluğunu temsil eder. Hiçbir production metriği sentetik referanstan doldurulmaz.

In [14]:
output_tables = {
    "production_table_inventory.csv": production_table_inventory,
    "motorcycle_service_coverage.csv": motorcycle_service_coverage,
    "service_interval_summary.csv": service_interval_summary,
    "mileage_quality_summary.csv": mileage_quality_summary,
    "censoring_summary.csv": censoring_summary,
    "tenant_feasibility_summary.csv": tenant_feasibility_summary,
    "production_part_category_inventory.csv": production_part_category_inventory,
    "part_category_mapping_candidates.csv": part_category_mapping_candidates,
    "production_service_todo_title_inventory.csv": production_service_todo_title_inventory,
    "production_service_todo_mapping_candidates.csv": production_service_todo_mapping_candidates,
    "production_mapping_coverage.csv": production_mapping_coverage,
    "production_feasibility_scorecard.csv": production_feasibility_scorecard,
}
for file_name, frame in output_tables.items():
    frame.to_csv(REPORT_TABLES_ROOT / file_name, index=False, encoding="utf-8-sig")

required_extract = '''# Required Production Extract — Issue #579

STATUS: BLOCKED until a read-only, anonymized production extract is supplied.

No production schema/export was available in the RideBase ML workspace. The following is the minimum semantic contract; source-native column names should be retained and documented in a manifest.

## motorcycles
- `motorcycle_id` — stable pseudonymous key
- `tenant_id` or `workshop_id` — pseudonymous tenant/workshop key
- `mileage` or current odometer
- `model` — if available
- `created_at`

## services
- `service_id` — stable pseudonymous key
- `motorcycle_id`
- `tenant_id` or `workshop_id`
- service date/timestamp
- service mileage/odometer
- `created_at`

## service_todos
- `todo_id`
- `service_id`
- title
- status — if available
- `created_at`

## part_categories
- `category_id`
- `tenant_id` or `workshop_id`
- category name

## Optional but useful
- `service_parts`, `parts`, `workshops`/`tenants`
- motorcycle brand/model/year
- maintenance policy or schedule fields

## Privacy and delivery rules
- Remove names, email, phone, address, VIN/chassis, plate, identity numbers and free-form personal notes.
- Keep stable pseudonymous entity and tenant IDs so joins and longitudinal ordering remain possible.
- Include an observation-window end timestamp and an export manifest with table row counts, schema, timezone and extraction time.
- Deliver CSV, Parquet, JSON or SQLite. SQLite will be opened read-only.
'''
(REPORTS_ROOT / "required_production_extract.md").write_text(required_extract, encoding="utf-8")

def fmt_metric(value, percent=False):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "N/A — BLOCKED"
    return f"{value:.2%}" if percent else str(value)

report_text = f'''# Executive Summary

**STATUS: {stage1_status}**  
Production data available: **{'YES' if production_available else 'NO'}**. {STATUS_REASON}. No synthetic v1.2 metrics are presented as production results.

# Data Availability

Minimum required entities (`motorcycles`, `services`, `service_todos`, `part_categories`): **{'AVAILABLE' if production_available else 'NOT AVAILABLE'}**.

# Service History

Motorcycles with >=2 services: {fmt_metric(coverage_metrics.get('motorcycles_2plus'))}.  
>=2 service rate: {fmt_metric(coverage_metrics.get('rate_2plus_all'), True)}.

# Mileage Quality

Services mileage fill: {fmt_metric(service_mileage_fill, True)}.  
Motorcycles current mileage fill: {fmt_metric(motorcycle_mileage_fill, True)}.

# Censoring

Snapshot and motorcycle-level definitions are documented separately. Metrics: {'computed from production' if not censoring_summary.empty else 'N/A — BLOCKED'}.

# Part Category Inventory

Production raw/normalized inventory: {'computed' if not production_part_category_inventory.empty else 'N/A — BLOCKED'}. Synthetic categories are reference vocabulary only.

# Service Todo Inventory

Raw unique titles: {fmt_metric(todo_metrics.get('raw_unique'))}.  
Normalized unique titles: {fmt_metric(todo_metrics.get('normalized_unique'))}.

# Canonical Mapping

High-confidence row coverage: {fmt_metric(mapped_row_rate, True)}.  
High-confidence unique-title coverage: {fmt_metric(mapped_unique_rate, True)}.

# Tenant Analysis

Tenant count: {fmt_metric(tenant_feasibility_summary['tenant_id'].nunique() if not tenant_feasibility_summary.empty else None)}. Tenant IDs are anonymized.

# Cold Start

<=1 service motorcycle rate: {fmt_metric(coverage_metrics.get('rate_le1'), True)}.

# ML Feasibility

NEXT SERVICE TIME: **{next_service_verdict}**  
NEXT TASK PREDICTION: **{next_task_verdict}**

# Recommended Architecture Direction

Obtain the read-only anonymized extract first. Then use leakage-safe survival/time-to-event ML for sufficiently observed histories, rule fallback for cold start, and deterministic normalization + exact/rule/fuzzy mapping with manual or optional LLM-assisted review for low-confidence text. LLM must not create label truth.

# Risks

- Missing production evidence blocks all quantitative conclusions.
- Tenant imbalance, sparse history, mileage inconsistency, label noise and multi-action titles remain unmeasured.
- Future service/todo/status/mileage fields must be excluded from prediction-time features.

# Final Verdict

NEXT SERVICE TIME: **{next_service_verdict}**  
NEXT TASK PREDICTION: **{next_task_verdict}**  
STAGE 1 STATUS: **{stage1_status}**
'''
(REPORTS_ROOT / "production_feasibility_report.md").write_text(report_text, encoding="utf-8")
print("CSV outputs:", len(output_tables))
print("Report:", REPORTS_ROOT / "production_feasibility_report.md")
print("Extract contract:", REPORTS_ROOT / "required_production_extract.md")

CSV outputs: 12
Report: /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase-ml/reports/production_feasibility_report.md
Extract contract: /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase-ml/reports/required_production_extract.md


# Inputs for Design Doc

- **Global vs tenant model:** Production tenant volumes ve drift ölçülmeden karar verilemez. Başlangıç adayı global model + tenant/workshop feature; tenant isolation ve minimum-volume gate zorunlu.
- **workshop_id feature:** Yalnız prediction anında biliniyor, privacy-safe ve leakage denetiminden geçiyorsa kullanılabilir; unseen-workshop fallback gerekir.
- **Tenant isolation:** Extract, training split, feature store, inference ve monitoring katmanlarında zorunlu.
- **Batch vs online inference:** Next-service/todo skorları için periyodik batch başlangıç adayıdır; online ancak operasyonel trigger ve latency gereksinimi doğrulanırsa.
- **Cold-start fallback:** Model maintenance policy + last service date/mileage + annual-km rule. Eksik alan varsa manuel/default policy.
- **Low-confidence handling:** Manual review queue; LLM yalnız suggestion/normalization assist, label truth değil.
- **Monitoring:** Tenant coverage, mileage fill/monotonicity, censoring, mapping coverage, unseen categories/titles, prediction drift ve calibration.
- **Retraining trigger:** Yeterli yeni observed next-event hacmi, drift eşiği veya mapping taxonomy değişimi.
- **Mapping versioning:** Canonical taxonomy, normalization rules, synonyms, thresholds ve reviewer kararları immutable version ile saklanmalı.

## Data leakage notu

Prediction snapshotından sonraki servis, completed todo, future mileage, future status, delivery/completion sonucu ve gelecekte oluşan mapping/reviewer kararı feature olarak kullanılamaz. Feature availability timestamp'i prediction timestamp'inden sonra olan tüm alanlar dışlanmalıdır.

# Issue #579 — Stage 1 Acceptance Checklist

Kanıt yoksa checkbox `DONE` yapılmaz. Aşağıdaki durumlar notebook tarafından production availability'ye göre oluşturulur.

In [15]:
acceptance_items = [
    ">=2 service motorcycle count/report", "service mileage completeness", "motorcycle mileage completeness",
    "censoring definition/rate", "part_categories inventory", "canonical category mapping draft",
    "service_todos title inventory", "service_todos clustering/mapping feasibility",
    "ML vs rule/LLM decision", "design doc inputs ready",
]
if production_available:
    acceptance_statuses = ["DONE", "DONE", "DONE", "DONE", "DONE", "DONE", "DONE", "DONE", "DONE", "DONE"]
else:
    acceptance_statuses = ["BLOCKED"] * 10
issue_579_acceptance = pd.DataFrame({"check": acceptance_items, "status": acceptance_statuses})
acceptance_counts = issue_579_acceptance["status"].value_counts().reindex(["DONE", "PARTIAL", "BLOCKED"], fill_value=0)
display(issue_579_acceptance)
display(acceptance_counts.rename_axis("status").reset_index(name="count"))

,check,status
0,>=2 service motorcycle count/report,BLOCKED
1,service mileage completeness,BLOCKED
2,motorcycle mileage completeness,BLOCKED
3,censoring definition/rate,BLOCKED
4,part_categories inventory,BLOCKED
5,canonical category mapping draft,BLOCKED
6,service_todos title inventory,BLOCKED
7,service_todos clustering/mapping feasibility,BLOCKED
8,ML vs rule/LLM decision,BLOCKED
9,design doc inputs ready,BLOCKED


,status,count
0,DONE,0
1,PARTIAL,0
2,BLOCKED,10


# Final Verdict

Production verisi olmadan Stage 1 kapanmış sayılmaz. Aşağıdaki hücre net, makine-okunabilir verdict üretir.

In [16]:
final_verdict = pd.DataFrame([
    {"decision": "NEXT SERVICE TIME", "verdict": next_service_verdict, "reason": "Production longitudinal service evidence unavailable" if not production_available else "See production metrics"},
    {"decision": "NEXT TASK PREDICTION", "verdict": next_task_verdict, "reason": "Production service_todos and mapping evidence unavailable" if not production_available else "See production metrics"},
    {"decision": "RECOMMENDED APPROACH", "verdict": "READ-ONLY EXTRACT → QA → HYBRID", "reason": "Survival ML where supported; rule fallback; deterministic text mapping + human/optional LLM assist"},
    {"decision": "STAGE 1 STATUS", "verdict": stage1_status, "reason": STATUS_REASON},
])
display(final_verdict)
print("Notebook completed without substituting synthetic metrics for production evidence.")

,decision,verdict,reason
0,NEXT SERVICE TIME,NOT_FEASIBLE,Production longitudinal service evidence unavailable
1,NEXT TASK PREDICTION,NOT_FEASIBLE,Production service_todos and mapping evidence unavailable
2,RECOMMENDED APPROACH,READ-ONLY EXTRACT → QA → HYBRID,Survival ML where supported; rule fallback; deterministic text mapping + human/optional LLM assist
3,STAGE 1 STATUS,BLOCKED,Production extract not available


Notebook completed without substituting synthetic metrics for production evidence.
